# Gene-body mCG vs expression (supporting)

Part of the **[Fig. 2 chapter](fig2.md)** — see it for the panel-by-panel map. The first code cell sets `ENTEX_ROOT` and activates the no-overwrite guard (see the [Reproduction guide](../reproduce.md)). *Outputs shown are the author's original run.*


## 📥 Required input files

This notebook reads the following files (paths resolve from `ENTEX_ROOT`/`REF_ROOT`; the setup cell sets them). See the chapter's `inputs.md` for RAW-vs-derived tags.

- `f'{indir}mcds/Pool_CZDA_PBMC.mcds'`  ·  _mC matrix (mcds)_
- `f'{indir}clustering/merged/5kCG100k3C_summary.h5ad'`  ·  _joint summary obj_
- `f'{indir}clustering/merged/L1pre/merge_5kCG_echo_entex_immune.h5ad'`  ·  _other_
- `'geneCG_expr_corr/PBMC_NaiveMem_gene_mCG.hdf'`  ·  _expression_
- `'PMD_ATAC/gene/bulkexpr_Hao2021.hdf'`  ·  _PMD/methyl-compartment_
- `'PMD_ATAC/gene/design_Hao2021.hdf'`  ·  _PMD/methyl-compartment_
- `f'{REF_ROOT}/hg38/gencode/v30/gencode.v30.annotation.gene.flat.tsv.gz'`  ·  _ref: gencode_
- `f'PMD_ATAC/gene/allgene.split50.slop250kb.5kb.{ct}.CGN-Merge.tsv'`  ·  _PMD/methyl-compartment_


In [1]:
# === Reproduction setup — edit ENTEX_ROOT / REF_ROOT for your machine ===
import os, sys
ENTEX_ROOT = os.environ.get('ENTEX_ROOT', '/large_storage/zhoulab/zhoujt/project/ENTEx')
REF_ROOT   = os.environ.get('REF_ROOT',   '/large_storage/zhoulab/ref')
BOOK_ROOT  = os.environ.get('BOOK_ROOT',  f'{ENTEX_ROOT}/analysis/HumanCellEpigenomeAtlas')
sys.path.insert(0, BOOK_ROOT)
os.chdir(f'{ENTEX_ROOT}/analysis')   # original working directory
import repro_guard                    # no-overwrite guard (default: skip existing)

In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt

import anndata
import scanpy as sc
import scanpy.external as sce
from sklearn.preprocessing import normalize
from sklearn.decomposition import TruncatedSVD

from ALLCools.clustering import *
from ALLCools.plot import *
from ALLCools.mcds import MCDS

import snapatac2 as snap
import warnings
warnings.filterwarnings("ignore")

mpl.style.use('default')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = 'Helvetica'


In [3]:
indir = f'{ENTEX_ROOT}/'

In [4]:
mcds = MCDS.open(f'{indir}mcds/Pool_CZDA_PBMC.mcds', var_dim='gene')
mcds

In [5]:
meta = anndata.read_h5ad(f'{indir}clustering/merged/5kCG100k3C_summary.h5ad').obs


In [6]:
adata_merge = anndata.read_h5ad(f'{indir}clustering/merged/L1pre/merge_5kCG_echo_entex_immune.h5ad')
adata_merge

In [7]:
selc = meta.index[meta.index.isin(adata_merge.obs.index)]
adata_merge.obs['celltype'] = adata_merge.obs['celltype'].astype(str)
adata_merge.obs[['L1', 'L2_any']] = meta[['L1', 'L2_any']].copy()
# adata_merge.obs.loc[selc, 'celltype'] = meta.loc[selc, 'leiden_cons'].values

In [8]:
ds = 0.5
coord_base = 'tsne'

fig, axes = plt.subplots(4, 3, figsize=(15, 15), dpi=300, constrained_layout=True)

tmp = adata_merge.obs.loc[adata_merge.obs['batch']=='echo']
ax = axes[0,0]
ax.scatter(adata_merge.obs[f'{coord_base}_0'], adata_merge.obs[f'{coord_base}_1'], c='#e0e0e0', edgecolors='none', s=ds, rasterized=True)
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='celltype',
                        text_anno='celltype',
                        s=ds*5,
                        labelsize=12,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        show_legend=True,
                        legend_kws={'ncol':1, 'fontsize':6},
                       )
for ax in axes[0, 1:]:
    ax.axis('off')
    
for i,ct in enumerate(['c1', 'c15', 'c7']):
    print(adata_merge.obs.loc[(adata_merge.obs['L1']==ct), 'L2_any'].unique().shape[0])
    for j in range(3):
        tmp = adata_merge.obs.loc[(adata_merge.obs['L1']==ct) & (adata_merge.obs['L2_any'].isin([f'c{k}' for k in np.arange(j*5, (j+1)*5)]))]
        ax = axes[i+1,j]
        ax.scatter(adata_merge.obs[f'{coord_base}_0'], adata_merge.obs[f'{coord_base}_1'], c='#e0e0e0', edgecolors='none', s=ds, rasterized=True)
        _ = categorical_scatter(data=tmp,
                                ax=ax,
                                coord_base=coord_base,
                                hue='L2_any',
                                text_anno='L2_any',
                                s=ds*2,
                                labelsize=12,
                                max_points=None,
                                palette='tab10',
                                scatter_kws={'rasterized':True},
                                show_legend=True)


In [9]:
leg = {'c1':{'Tc-Mem':[1,2,3,5,9,10,11,13,14], 'Th-Mem':[0,4,6,7,8,12]},
       'c15':{'Tc-Naive':[3,4], 'Th-Naive':[0,1,2,5,6,7,8,9,10,11,12]},
       'c7':{'B-Naive':[0,10,11], 'B-Mem':[1,2,3,4,5,6,7,8,9,12]}}


In [10]:
meta_hema = meta.loc[meta['L1'].isin(leg.keys())]
meta_hema['celltype'] = ''
for l1 in leg:
    for ct in leg[l1]:
        selct = [f'c{i}' for i in leg[l1][ct]]
        selc = (meta_hema['L1']==l1) & (meta_hema['L2_any'].isin(selct))
        meta_hema.loc[selc, 'celltype'] = ct
        

In [11]:
meta_hema['cell_group'] = meta_hema['celltype'].astype(str) + '-' + meta_hema['Tissue'].astype(str)

In [12]:
meta_hema_pbmc = meta_hema.loc[(meta_hema['Tissue']=='PBMC') & (meta_hema['L1'].isin(leg.keys()))]


In [13]:
selc = mcds.get_index('cell').intersection(meta_hema_pbmc.index)
mcds = mcds.sel({'cell':selc})

In [14]:
mcds = mcds.assign_coords(celltype=('cell', meta_hema_pbmc.loc[mcds.get_index('cell'), 'celltype']))


In [15]:
mcds = mcds.groupby('celltype').sum()
mcds = MCDS(mcds, obs_dim='celltype', var_dim='gene')


In [16]:
mcds = mcds.sel({'mc_type':'CGN'})

In [17]:
cov = mcds['gene_da'].sel(count_type='cov').mean(dim='celltype').squeeze().to_pandas()


In [18]:
mcds = mcds.sel({'gene':cov.index[cov>100]})

black_list_path = f'{REF_ROOT}/blacklist/hg38-blacklist.v2.bed.gz'
mcds = mcds.remove_black_list_region(
    black_list_path=black_list_path, f=0.5
)

exclude_chromosome = ['chrX', 'chrY', 'chrM', 'chrL']
mcds = mcds.remove_chromosome(exclude_chromosome)


In [19]:
# mcds.add_mc_frac(normalize_per_cell=False)
mcds['gene_da_frac'] = mcds['gene_da'].sel({'count_type':'mc'}) / mcds['gene_da'].sel({'count_type':'cov'})
data = mcds['gene_da_frac'].to_pandas()


In [20]:
data.to_hdf('geneCG_expr_corr/PBMC_NaiveMem_gene_mCG.hdf', key='data')
